In [1]:
import pandas as pd
import numpy as np
from pmdarima import auto_arima
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)
data = data.set_index("time")
data.index.freq = "h"

split = int(np.ceil(0.8 * len(data)))
train = data.iloc[:split]
test  = data.iloc[split:]

# --- Train ---
model = auto_arima(
    train["pm2_5"],
    seasonal=False,
    stepwise=True,
    information_criterion="aic",
    max_p=2, max_q=2
)

# --- Rolling 72-hour-ahead evaluation ---
HORIZON = 72
test_pm25 = test["pm2_5"].reset_index(drop=True)

preds = []
for i in range(len(test_pm25) - HORIZON + 1):
    forecast = np.asarray(model.predict(n_periods=HORIZON))
    preds.append(forecast[-1])
    model.update(test_pm25.iloc[[i]])

preds = np.array(preds)
true = test_pm25.iloc[HORIZON - 1 : HORIZON - 1 + len(preds)].values

rmse = root_mean_squared_error(true, preds)
mae  = mean_absolute_error(true, preds)
r2   = r2_score(true, preds)

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")

RMSE : 22.6420
MAE  : 15.5698
R2   : 0.4292
